# Baseline — Animal Voice Detection

**Competition:** classify 5-second animal sound clips into 10 anonymous classes. Each
clip comes both as audio (`.wav`) and as a **mel-spectrogram image** — so you can treat
audio classification as *image* classification.

- **Task:** 10-class classification (240 train / 160 test clips, balanced)
- **Metric:** accuracy
- **Kaggle link:** _TODO: add link_

**Approach:** frozen ImageNet ResNet-18 features on the spectrogram images + Logistic
Regression.

In [1]:
import numpy as np
import pandas as pd
import torch
from torchvision.models import resnet18, ResNet18_Weights
from PIL import Image

DATA_DIR = "."
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
print(train.shape, test.shape)

(240, 4) (160, 3)


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
weights = ResNet18_Weights.IMAGENET1K_V1
backbone = resnet18(weights=weights); backbone.fc = torch.nn.Identity()
backbone.eval().to(device)
preprocess = weights.transforms()

@torch.no_grad()
def extract(paths, bs=32):
    out = []
    for i in range(0, len(paths), bs):
        b = [preprocess(Image.open(f"{DATA_DIR}/{p}").convert("RGB")) for p in paths[i:i+bs]]
        out.append(backbone(torch.stack(b).to(device)).cpu().numpy())
    return np.vstack(out)

X  = extract(train["spectrogram_path"].tolist())
Xt = extract(test["spectrogram_path"].tolist())
print(X.shape, Xt.shape)

(240, 512) (160, 512)


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

y = train["label"].values
clf = LogisticRegression(max_iter=3000)
scores = cross_val_score(clf, X, y, cv=5)
print(f"CV accuracy: {scores.mean():.4f} +/- {scores.std():.4f}")

CV accuracy: 0.5958 +/- 0.0250


In [4]:
clf.fit(X, y)
sub = pd.DataFrame({"id": test["id"], "label": clf.predict(Xt)})
sub.to_csv("submission.csv", index=False)
sub.head()

,id,label
0,test_0000,animal_00
1,test_0001,animal_05
2,test_0002,animal_07
3,test_0003,animal_08
4,test_0004,animal_08


## Ideas to improve

- Work on the **raw audio**: log-mel with different resolutions, or embeddings from a
  pretrained audio model (PANNs, AST, or `torchaudio` wav2vec2).
- Augment audio: time-shift, noise, SpecAugment (mask time/frequency stripes).
- Fine-tune the CNN on GPU; 240 clips is tiny, so augmentation matters a lot.
